In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [2]:
import subprocess
import sys
import os

print("Starting consolidated dependency installation...")

# Upgrade pip first to ensure it's up-to-date
try:
    print("Upgrading pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    print("pip upgraded successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error upgrading pip: {e}")
    print("Continuing with installation, but consider manually updating pip if issues persist.")

# Clear pip cache to ensure fresh downloads
try:
    print("Clearing pip cache...")
    subprocess.check_call([sys.executable, "-m", "pip", "cache", "purge"])
    print("Pip cache cleared.")
except subprocess.CalledProcessError as e:
    print(f"Error clearing pip cache: {e}")

# This subprocess setup is crucial for fixing Google Colab's common `numpy` conflicts.
# The `numpy.dtype size changed` error often indicates binary incompatibility
# with pre-installed PyTorch/TorchVision when `numpy` is implicitly updated by other packages.
# By installing all core dependencies here and *avoiding* explicit `numpy` and `scipy`
# uninstallation/installation, we allow pip to resolve compatibility and rely on
# Colab's default `numpy`/`scipy` that are compatible with its `Torch`/`TorchVision` setup.
packages_to_install = [
    "albumentations==2.0.8", # Updated to the version installed later in the notebook
    "grad-cam==1.5.5", # Pinning to the version that was installed later in the notebook
    "ttach",
    "opencv-python", # Explicitly install opencv-python as used with cv2
    "matplotlib", # Explicitly install matplotlib as used for plotting
    "gradio", # Added gradio for the UI part
    "rembg[cpu]" # Re-adding rembg[cpu] to ensure onnxruntime backend is installed
]

try:
    print(f"Installing {', '.join(packages_to_install)}...")
    # Use --upgrade to ensure they are the latest available if already present
    # Use --no-cache-dir to ensure fresh download, combined with cache purge earlier
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir"] + packages_to_install)
    print("All specified dependencies installed successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error during installation of core dependencies: {e}")
    print("Please check pip logs for details. Critical dependencies might be missing or incompatible.")
    sys.exit(1)

print("Dependency setup complete. You may now proceed with the rest of the notebook.")

Starting consolidated dependency installation...
Upgrading pip...
pip upgraded successfully.
Clearing pip cache...
Pip cache cleared.
Installing albumentations==2.0.8, grad-cam==1.5.5, ttach, opencv-python, matplotlib, gradio, rembg[cpu]...
All specified dependencies installed successfully.
Dependency setup complete. You may now proceed with the rest of the notebook.


In [3]:
import subprocess
import sys

# Reinstall torch and torchvision to resolve potential numpy binary incompatibility.
# This is often necessary in environments like Colab where numpy might get updated
# by other packages, leading to conflicts with pre-compiled torchvision.
print("Attempting to reinstall torch and torchvision to resolve potential numpy conflicts...")
try:
    # Use --force-reinstall to ensure they pick up the current numpy environment.
    # Use --upgrade to get the latest compatible versions.
    # The `--index-url` ensures we fetch the correct CUDA-enabled versions for Colab's GPU.
    # (Assuming CUDA 12.1, which is common for Colab's T4 GPUs. Adjust if needed for a different CUDA version).
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall", "--upgrade", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"])
    print("torch and torchvision reinstalled successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error reinstalling torch and torchvision: {e}")
    print("Continuing, but expect potential issues if the conflict is not resolved.")

Attempting to reinstall torch and torchvision to resolve potential numpy conflicts...
torch and torchvision reinstalled successfully.


In [4]:
import os

# Define the local SSD target directory
LOCAL_DATA_DIR = '/tmp/Datasets_367v_Cleaned_224'

# Create the target directory if it doesn't exist
!mkdir -p {LOCAL_DATA_DIR}

# Copy the dataset from Google Drive to local SSD
# The -r flag ensures recursive copying for directories
# The -L flag (or --dereference) will follow symbolic links, if any
# Using `rsync -av` is generally more robust for large directories if available
print(f"Copying dataset from Google Drive to {LOCAL_DATA_DIR}...")
# Reverting to copy the *contents* of Datasets_3c into LOCAL_DATA_DIR
!cp -r /content/drive/MyDrive/Datasets_367v_Cleaned_224/* {LOCAL_DATA_DIR}/
print("Dataset copied to local SSD.")

print(f"Contents of {LOCAL_DATA_DIR} after copy:")
!ls -F {LOCAL_DATA_DIR}

Copying dataset from Google Drive to /tmp/Datasets_367v_Cleaned_224...
Dataset copied to local SSD.
Contents of /tmp/Datasets_367v_Cleaned_224 after copy:
01_Train/  02_Val/  03_Test/


In [5]:
# JUST CHECKING
print(f"Contents of Google Drive source directory: /content/drive/MyDrive/Datasets_367v_Cleaned_224")
!ls -F /content/drive/MyDrive/Datasets_367v_Cleaned_224

Contents of Google Drive source directory: /content/drive/MyDrive/Datasets_367v_Cleaned_224
01_Train/  02_Val/  03_Test/


In [6]:
import os

# Define the local SSD target directory (assuming LOCAL_DATA_DIR is defined in cell 9c82739e)
LOCAL_DATA_DIR = '/tmp/Datasets_367v_Cleaned_224'

# Update directory paths to point to the original local SSD dataset
TRAIN_DIR = os.path.join(LOCAL_DATA_DIR, '01_Train')
VAL_DIR = os.path.join(LOCAL_DATA_DIR, '02_Val')
TEST_DIR = os.path.join(LOCAL_DATA_DIR, '03_Test')

print(f"Updated TRAIN_DIR: {TRAIN_DIR}")
print(f"Updated VAL_DIR: {VAL_DIR}")
print(f"Updated TEST_DIR: {TEST_DIR}")

Updated TRAIN_DIR: /tmp/Datasets_367v_Cleaned_224/01_Train
Updated VAL_DIR: /tmp/Datasets_367v_Cleaned_224/02_Val
Updated TEST_DIR: /tmp/Datasets_367v_Cleaned_224/03_Test


In [7]:
import sys

import os
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import mobilenet_v3_large

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

# **^^ Declare Variables ^^**

In [8]:
import os

# General Configuration
IMAGE_SIZE = 224
BATCH_SIZE = 32

# Ensure TRAIN_DIR is defined. It should be from cell f53f7937.
if 'TRAIN_DIR' not in globals():
    # Fallback/warning if TRAIN_DIR isn't defined, though it should be.
    print("Warning: TRAIN_DIR not found in global scope. Attempting to define from default /tmp path.")
    LOCAL_DATA_DIR = '/tmp/Datasets_367v_Cleaned_224'
    TRAIN_DIR = os.path.join(LOCAL_DATA_DIR, '01_Train')
    if not os.path.isdir(TRAIN_DIR):
        raise ValueError(f"TRAIN_DIR ({TRAIN_DIR}) is not a valid directory. Please run previous cells to ensure data is copied correctly.")

print(f"Dynamically determining CLASS_NAMES from {TRAIN_DIR}...")
# Get all entries in TRAIN_DIR that are directories, exclude hidden files
all_entries = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
CLASS_NAMES = sorted([entry for entry in all_entries if not entry.startswith('.')])

if not CLASS_NAMES:
    raise ValueError(f"No class directories found in {TRAIN_DIR}. Please check your dataset structure.")

NUM_CLASSES = len(CLASS_NAMES)

print(f"Detected CLASS_NAMES: {CLASS_NAMES}")
print(f"NUM_CLASSES: {NUM_CLASSES}")

Dynamically determining CLASS_NAMES from /tmp/Datasets_367v_Cleaned_224/01_Train...
Detected CLASS_NAMES: ['FaLan', 'Keaw_Sawoey', 'NamDokMai_Seethong', 'None']
NUM_CLASSES: 4


In [9]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 2. Albumentations & OpenCV Pipeline

# Define the transformations using Albumentations
train_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE, interpolation=cv2.INTER_LANCZOS4),
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=128, p=1.0),
    A.HorizontalFlip(p=0.5),
    # A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.6),
    # A.RandomSunFlare(flare_roi=(0, 0, 1, 0.6), src_radius=120, p=0.4),
    # A.RandomShadow(shadow_dimension=5, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), # Normalize after augmentations
    ToTensorV2()
])

val_test_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE, interpolation=cv2.INTER_LANCZOS4),
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=128, p=1.0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class MangoLeafDataset(Dataset):
    def __init__(self, root_dir, class_names, transform=None):
        self.root_dir = root_dir
        self.class_names = class_names
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.label_map = {name: i for i, name in enumerate(class_names)}

        for class_name in class_names:
            class_path = os.path.join(root_dir, class_name)
            # Check if directory exists before listing its contents
            if not os.path.isdir(class_path):
                raise FileNotFoundError(f"Class directory not found: {class_path}. Please check your dataset path and Google Drive mount.")
            for img_name in os.listdir(class_path):
                self.image_paths.append(os.path.join(class_path, img_name))
                self.labels.append(self.label_map[class_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Read image dynamically from disk via OpenCV
        image = cv2.imread(img_path)
        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label, img_path

/tmp/ipykernel_7634/62418823.py:10: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=128, p=1.0),
/tmp/ipykernel_7634/62418823.py:21: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=128, p=1.0),


In [10]:
# Create Dataset and DataLoader instances
# CLASS_NAMES and NUM_CLASSES are now dynamically defined in an earlier cell (eca6b32f).
# This ensures consistency with the actual dataset directories found on disk.

train_dataset = MangoLeafDataset(TRAIN_DIR, CLASS_NAMES, train_transforms)
val_dataset = MangoLeafDataset(VAL_DIR, CLASS_NAMES, val_test_transforms)
test_dataset = MangoLeafDataset(TEST_DIR, CLASS_NAMES, val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

Number of training samples: 1703
Number of validation samples: 236
Number of test samples: 223


### Verify Training Directory Contents

Let's explicitly list the contents of the TRAIN_DIR to confirm the presence and exact naming of the class folders, especially 'None'.

In [11]:
print(f"Listing contents of the training directory: {TRAIN_DIR}")
!ls -F {TRAIN_DIR}

print(f"\nListing contents of the validation directory: {VAL_DIR}")
!ls -F {VAL_DIR}

print(f"\nListing contents of the test directory: {TEST_DIR}")
!ls -F {TEST_DIR}


Listing contents of the training directory: /tmp/Datasets_367v_Cleaned_224/01_Train
FaLan/	Keaw_Sawoey/  NamDokMai_Seethong/  None/

Listing contents of the validation directory: /tmp/Datasets_367v_Cleaned_224/02_Val
FaLan/	Keaw_Sawoey/  NamDokMai_Seethong/  None/

Listing contents of the test directory: /tmp/Datasets_367v_Cleaned_224/03_Test
FaLan/	Keaw_Sawoey/  NamDokMai_Seethong/  None/


In [12]:
from collections import Counter
import torch

# Define the device here so it's available for class_weights_tensor
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device for class weights: {device}")

# Get class counts from the training dataset
# train_dataset is created in cell 8e491f0c, which depends on CLASS_NAMES from eca6b32f.
# Ensure eca6b32f and 8e491f0c are executed first.
class_counts = Counter(train_dataset.labels)

# Sort class counts by class index to ensure correct order
# Handle cases where a class might have zero samples (though ideally, all classes should be represented)
sorted_class_counts = [class_counts[i] for i in range(len(CLASS_NAMES))]

# Calculate inverse class weights based on frequency
# Handle cases where count is zero to avoid ZeroDivisionError
total_samples = sum(sorted_class_counts)
base_class_weights = [total_samples / (NUM_CLASSES * (count if count > 0 else 1)) for count in sorted_class_counts]


# Define manual adjustment factors. This list needs to match the length of CLASS_NAMES.
# Initialize with neutral factors (1.0) for all classes, then apply specific adjustments.
manual_adjustment_factors = [1.0] * NUM_CLASSES

# Apply specific adjustments if the class is present
if 'Keaw_Sawoey' in CLASS_NAMES:
    manual_adjustment_factors[CLASS_NAMES.index('Keaw_Sawoey')] = 1.2 # Increase for 'Keaw_Sawoey'
if 'FaLan' in CLASS_NAMES:
    manual_adjustment_factors[CLASS_NAMES.index('FaLan')] = 1.1 # Increase for 'FaLan'
# 'NamDokMai_Seethong' and 'None' will retain the default 1.0 factor unless specified otherwise.

# Apply manual adjustments to the base weights
class_weights = [base_class_weights[i] * manual_adjustment_factors[i] for i in range(NUM_CLASSES)]

# Convert to a PyTorch tensor and move to the device
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class distribution in training dataset:", class_counts)
print("Calculated base class weights (inverse frequency):")
for i, w in enumerate(base_class_weights):
    print(f"  {CLASS_NAMES[i]}: {w:.4f}")
print("Manual adjustment factors:", manual_adjustment_factors)
print("Final adjusted class weights:", class_weights_tensor)

Using device for class weights: cpu
Class distribution in training dataset: Counter({2: 539, 0: 407, 1: 403, 3: 354})
Calculated base class weights (inverse frequency):
  FaLan: 1.0461
  Keaw_Sawoey: 1.0565
  NamDokMai_Seethong: 0.7899
  None: 1.2027
Manual adjustment factors: [1.1, 1.2, 1.0, 1.0]
Final adjusted class weights: tensor([1.1507, 1.2677, 0.7899, 1.2027])


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v3_large
from torch.optim.lr_scheduler import ReduceLROnPlateau # New import for scheduler

# CLASS_NAMES and NUM_CLASSES are now dynamically defined in cell eca6b32f.
# Ensure that cell eca6b32f has been executed before this cell.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Use CUDA if available
print(f"Using device: {device}")

# Load pre-trained MobileNetV3 Large model
model = mobilenet_v3_large(pretrained=True)

# Freeze all parameters initially
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier head with a new one for NUM_CLASSES
# Parameters in this new head will have requires_grad=True by default
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[0].in_features, 512),
    nn.Hardswish(),
    nn.Dropout(p=0.4),
    nn.Linear(512, NUM_CLASSES) # Updated to NUM_CLASSES (dynamically set)
)

# Unfreeze the very last layer of model.features (the last block in the feature extractor)
for param in model.features[-1].parameters():
    param.requires_grad = True

# Move model to the appropriate device
model.to(device)

# Define Loss Function with custom class weights
# class_weights_tensor is expected to be defined in a preceding cell (9b362c70)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Define Optimizer: Only update parameters that require gradients (classifier head and last feature layer)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5) # Reduced learning rate

# Add learning rate scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

print("Model configured for Partial Fine-tuning:")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth
100%|██████████| 21.1M/21.1M [00:00<00:00, 89.4MB/s]


Model configured for Partial Fine-tuning:
Total parameters: 3466036
Trainable parameters: 649604


  # **#         END OF BASELINES AND AUGMENTATION**